## 🎯 Learning Objectives
* Demonstrate a comprehensive understanding of LangChain 0.3 architecture and its core components.
* Apply LCEL principles to construct robust and efficient LLM chains and pipelines.
* Implement and evaluate Retrieval Augmented Generation (RAG) systems for knowledge retrieval.
* Design, build, and interact with LangChain agents capable of using multiple tools and maintaining conversational state.
* Integrate various LangChain modules (LLMs, Prompts, Parsers, Tools, Memory, Retrievers) into a cohesive application.


## LLM02-FA: Final Assessment - Introduction to LangChain for Agentic AI

Welcome to the final assessment for "LLM-02: Introduction to LangChain for Agentic AI"! This assessment is designed to evaluate your comprehensive understanding and practical application of the concepts covered throughout the course, including LangChain 0.3 architecture, LCEL pipelines, retrieval chains, and the development of your first LangChain agent.

In the rapidly evolving landscape of AI in 2026, proficiency in frameworks like LangChain is paramount for building sophisticated, agentic LLM applications. This assessment will challenge you to not only recall theoretical knowledge but also to apply it in a practical, hands-on coding scenario.

**What this assessment covers:**

*   **LangChain 0.3 Core Concepts:** Understanding the modularity, `Runnable` interface, and the power of LCEL.
*   **Prompt Engineering with LangChain:** Crafting effective prompts and managing conversational history.
*   **Retrieval Augmented Generation (RAG):** Implementing systems to ground LLM responses in external knowledge bases.
*   **Tools and Agents:** Equipping LLMs with external capabilities and orchestrating complex multi-step reasoning.
*   **Memory Management:** Maintaining conversational context for stateful interactions.

This assessment is structured into two main parts: a set of review questions to test your conceptual understanding, and a capstone coding project that requires you to integrate various LangChain components to solve a real-world problem. Good luck!


### Part 1: Review Questions

Answer the following questions concisely, demonstrating your understanding of LangChain's core principles and components.

1.  **LangChain 0.3 Architecture:** Describe the primary benefits of the `Runnable` interface and the LCEL (LangChain Expression Language) for building LLM applications, especially in comparison to earlier versions of LangChain or direct API calls.

2.  **LCEL Pipelines:** Explain how the `|` (pipe) operator works in LCEL. Provide a simple example demonstrating its use with at least two `Runnable` components.

3.  **Retrieval Augmented Generation (RAG):** What problem does RAG primarily solve for LLMs? Outline the key components of a typical RAG chain in LangChain.

4.  **Tools vs. Chains:** Differentiate between a "Chain" and a "Tool" in the context of LangChain agents. When would you choose to implement functionality as a Tool rather than a standalone Chain?

5.  **Agent Components:** List and briefly describe the essential components required to construct a functional LangChain agent using `AgentExecutor`.

6.  **Memory in Agents:** Why is memory crucial for agentic applications? Describe how `ConversationBufferWindowMemory` works and when it would be preferred over `ConversationBufferMemory`.

7.  **Custom Tools:** Imagine you need to create a tool that interacts with a proprietary internal API. Outline the steps you would take to define and integrate this custom tool into a LangChain agent.

8.  **Output Parsers:** Explain the role of `StrOutputParser` and `JsonOutputParser` in LCEL pipelines. Provide a scenario where `JsonOutputParser` would be indispensable.


### Part 2: Capstone Project - Enterprise Knowledge Navigator Agent

**Scenario:**

Your organization, "InnovateCorp," has a vast internal knowledge base consisting of various documents (HR policies, project specifications, technical guides, meeting notes). Employees frequently need to query these documents for information, but also sometimes need to search the web for current industry trends or external definitions. Your task is to build an intelligent **Enterprise Knowledge Navigator Agent** using LangChain that can:

1.  **Answer questions about internal InnovateCorp documents:** The agent should be able to retrieve relevant information from a simulated internal knowledge base (vector store).
2.  **Perform external web searches:** If the internal knowledge base cannot answer a question, or if the query explicitly asks for external information, the agent should use a web search tool.
3.  **Summarize internal documents:** If a user asks for a summary of a specific document or a topic covered in the internal docs, the agent should be able to provide it.
4.  **Maintain conversational context:** The agent should remember previous turns in the conversation to answer follow-up questions.

**Requirements:**

*   Utilize **LangChain 0.3+** and **LCEL** for building the agent and its components.
*   Use an **OpenAI-compatible LLM** (e.g., `ChatOpenAI`).
*   Implement a **vector store** (e.g., `Chroma`, `FAISS`) to simulate InnovateCorp's internal knowledge base. Populate it with at least 3-5 dummy documents.
*   Create a **custom tool** for searching the internal knowledge base.
*   Integrate a **web search tool** (e.g., `TavilySearchResults`).
*   Configure the agent with **memory** to handle conversational history.
*   Demonstrate the agent's capabilities with at least **three distinct queries** that showcase:
    *   Retrieval from internal documents.
    *   Use of the web search tool.
    *   A follow-up question leveraging conversational memory.

**Evaluation Criteria:**

*   Correct implementation of LangChain components.
*   Effective use of LCEL for chaining.
*   Proper tool definition and integration.
*   Successful handling of conversational memory.
*   Clear and well-commented code.
*   Agent's ability to correctly respond to the specified query types.


In [ ]:
# Part 2: Capstone Project - Enterprise Knowledge Navigator Agent

# Install necessary libraries (if not already installed)
# !pip install -q langchain langchain-openai langchain-chroma tavily-python

import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain import hub
from langchain_core.tools import tool
from langchain_community.tools.tavily_research import TavilySearchResults
from langchain.memory import ConversationBufferWindowMemory

# --- Configuration --- #
# Set your API keys here. Replace 'YOUR_OPENAI_API_KEY' and 'YOUR_TAVILY_API_KEY'
# It's recommended to load these from environment variables for security.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# --- 1. Simulate InnovateCorp's Internal Knowledge Base (Vector Store) ---
# Create dummy documents for the internal knowledge base
internal_docs = [
    Document(page_content="InnovateCorp's HR policy on remote work states that employees can work remotely up to 3 days a week, subject to manager approval and team requirements. A formal request must be submitted via the HR portal.", metadata={"source": "HR Policy Manual"}),
    Document(page_content="Project 'Quantum Leap' aims to develop a new AI-powered analytics platform by Q4 2026. Key technologies include federated learning, edge computing, and a custom LLM fine-tuned for financial data. The project lead is Dr. Anya Sharma.", metadata={"source": "Project Quantum Leap Spec"}),
    Document(page_content="The company's Q3 2026 financial report shows a 15% increase in revenue, primarily driven by the success of the 'Synapse' product line. Net profit increased by 10% year-over-year. The next board meeting is scheduled for October 25, 2026.", metadata={"source": "Q3 2026 Financial Report"}),
    Document(page_content="Our new employee onboarding process involves a 2-week orientation program, mandatory compliance training, and a mentorship assignment. New hires receive their equipment on their first day.", metadata={"source": "Onboarding Guide"}),
    Document(page_content="InnovateCorp's core values are Innovation, Integrity, Collaboration, and Customer Focus. These values guide all our business decisions and employee interactions.", metadata={"source": "Company Values Statement"})
]

# Create a Chroma vector store from the documents
vectorstore = Chroma.from_documents(documents=internal_docs, embedding=embeddings)
retriever = vectorstore.as_retriever()

# --- 2. Define Custom Tool for Internal Knowledge Base Search ---
@tool
def internal_knowledge_search(query: str) -> str:
    """Searches InnovateCorp's internal knowledge base for relevant information. Use this tool for questions about company policies, projects, financial reports, or internal processes."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant internal documents found."
    return "\n\n".join([doc.page_content for doc in docs])

# --- 3. Integrate Web Search Tool ---
tavily_tool = TavilySearchResults(max_results=3)

# --- 4. Combine Tools ---
tools = [internal_knowledge_search, tavily_tool]

# --- 5. Configure Agent with Memory ---
# Pull the standard OpenAI functions agent prompt from LangChain Hub
prompt = hub.pull("hwchase17/openai-functions-agent")

# Initialize memory
memory = ConversationBufferWindowMemory(memory_key="chat_history", return_messages=True, k=5)

# Create the agent
agent = create_openai_functions_agent(llm, tools, prompt)

# Create the Agent Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True # Important for robust agents
)

# --- 6. Demonstrate Agent Capabilities ---
print("\n--- Agent Interaction Start ---")

# Query 1: Retrieval from internal documents
print("\nUser: What is InnovateCorp's policy on remote work?")
response1 = agent_executor.invoke({"input": "What is InnovateCorp's policy on remote work?"})
print(f"Agent: {response1['output']}")

# Query 2: Use of the web search tool (explicitly external)
print("\nUser: What are the latest trends in federated learning in 2026?")
response2 = agent_executor.invoke({"input": "What are the latest trends in federated learning in 2026?"})
print(f"Agent: {response2['output']}")

# Query 3: A follow-up question leveraging conversational memory
print("\nUser: Who is the project lead for that AI analytics platform?")
response3 = agent_executor.invoke({"input": "Who is the project lead for that AI analytics platform?"})
print(f"Agent: {response3['output']}")

# Query 4: Another internal query, potentially requiring summarization or specific detail
print("\nUser: Can you tell me about the Q3 2026 financial report? What was the revenue increase?")
response4 = agent_executor.invoke({"input": "Can you tell me about the Q3 2026 financial report? What was the revenue increase?"})
print(f"Agent: {response4['output']}")

print("\n--- Agent Interaction End ---")


In [ ]:
# Detailed Solution Code for Capstone Project

# Ensure you have the necessary libraries installed:
# !pip install -q langchain langchain-openai langchain-chroma tavily-python

import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain import hub
from langchain_core.tools import tool
from langchain_community.tools.tavily_research import TavilySearchResults
from langchain.memory import ConversationBufferWindowMemory

# --- Configuration --- #
# Set your API keys here. For production, always use environment variables.
# You can set them directly for this assessment if preferred, but be cautious.
# Example:
# os.environ["OPENAI_API_KEY"] = "sk-YOUR_OPENAI_API_KEY_HERE"
# os.environ["TAVILY_API_KEY"] = "tvly-YOUR_TAVILY_API_KEY_HERE"

# Initialize the LLM for agent reasoning and response generation
# Using gpt-4o-mini for cost-effectiveness and good performance in 2026.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Initialize embeddings for the vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# --- 1. Simulate InnovateCorp's Internal Knowledge Base (Vector Store) ---
# Create a list of Document objects to represent internal company knowledge.
# Each document has page_content (the text) and metadata (e.g., source).
internal_docs = [
    Document(page_content="InnovateCorp's HR policy on remote work states that employees can work remotely up to 3 days a week, subject to manager approval and team requirements. A formal request must be submitted via the HR portal.", metadata={"source": "HR Policy Manual"}),
    Document(page_content="Project 'Quantum Leap' aims to develop a new AI-powered analytics platform by Q4 2026. Key technologies include federated learning, edge computing, and a custom LLM fine-tuned for financial data. The project lead is Dr. Anya Sharma.", metadata={"source": "Project Quantum Leap Spec"}),
    Document(page_content="The company's Q3 2026 financial report shows a 15% increase in revenue, primarily driven by the success of the 'Synapse' product line. Net profit increased by 10% year-over-year. The next board meeting is scheduled for October 25, 2026.", metadata={"source": "Q3 2026 Financial Report"}),
    Document(page_content="Our new employee onboarding process involves a 2-week orientation program, mandatory compliance training, and a mentorship assignment. New hires receive their equipment on their first day.", metadata={"source": "Onboarding Guide"}),
    Document(page_content="InnovateCorp's core values are Innovation, Integrity, Collaboration, and Customer Focus. These values guide all our business decisions and employee interactions.", metadata={"source": "Company Values Statement"}),
    Document(page_content="The 'Synapse' product line, launched in early 2026, leverages advanced neural networks for real-time market prediction and has been a significant revenue driver. It integrates seamlessly with existing financial systems.", metadata={"source": "Product Strategy Document"})
]

# Create a Chroma vector store from the dummy documents.
# Chroma is an in-memory vector store suitable for this example.
vectorstore = Chroma.from_documents(documents=internal_docs, embedding=embeddings)

# Create a retriever from the vector store. This will be used by our custom tool.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # Retrieve top 3 relevant documents

# --- 2. Define Custom Tool for Internal Knowledge Base Search ---
# We use the @tool decorator to easily convert a Python function into a LangChain tool.
@tool
def internal_knowledge_search(query: str) -> str:
    """Searches InnovateCorp's internal knowledge base for relevant information.
    Use this tool for questions about company policies, projects, financial reports, internal processes, or company values.
    Returns the content of relevant documents."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant internal documents found in InnovateCorp's knowledge base."
    # Concatenate the page content of the retrieved documents.
    return "\n\n".join([f"Source: {doc.metadata.get('source', 'Unknown')}\nContent: {doc.page_content}" for doc in docs])

# --- 3. Integrate Web Search Tool ---
# TavilySearchResults is a powerful web search tool. Max_results limits the number of search results.
# Ensure TAVILY_API_KEY is set in your environment or directly in the code.
tavily_tool = TavilySearchResults(max_results=3)

# --- 4. Combine Tools ---
# The agent will have access to both the internal knowledge search and the external web search.
tools = [internal_knowledge_search, tavily_tool]

# --- 5. Configure Agent with Memory ---
# Pull a standard prompt for OpenAI function calling agents from LangChain Hub.
# This prompt is designed to work well with create_openai_functions_agent.
prompt = hub.pull("hwchase17/openai-functions-agent")

# Initialize memory for the agent.
# ConversationBufferWindowMemory keeps a window of the last 'k' messages,
# preventing the memory from growing indefinitely and managing context.
# `memory_key="chat_history"` is crucial as the prompt expects this key.
# `return_messages=True` ensures the history is returned as a list of message objects.
memory = ConversationBufferWindowMemory(memory_key="chat_history", return_messages=True, k=5)

# Create the agent using create_openai_functions_agent.
# This agent type is specifically designed to leverage OpenAI's function calling capabilities.
agent = create_openai_functions_agent(llm, tools, prompt)

# Create the Agent Executor.
# The AgentExecutor is responsible for running the agent, managing tools, and handling memory.
# `verbose=True` prints the agent's thought process, which is very useful for debugging.
# `handle_parsing_errors=True` makes the agent more robust to occasional LLM output errors.
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True
)

# --- 6. Demonstrate Agent Capabilities ---
print("\n--- InnovateCorp Knowledge Navigator Agent - Interaction Start ---")
print("Hello! I am your Enterprise Knowledge Navigator. How can I assist you today?")

# Query 1: Retrieval from internal documents
# The agent should use 'internal_knowledge_search' tool.
user_query_1 = "What is InnovateCorp's policy on remote work?"
print(f"\nUser: {user_query_1}")
response1 = agent_executor.invoke({"input": user_query_1})
print(f"Agent: {response1['output']}")

# Query 2: Use of the web search tool (explicitly external)
# The agent should identify that this requires external knowledge and use 'tavily_tool'.
user_query_2 = "What are the latest advancements in quantum computing in 2026?"
print(f"\nUser: {user_query_2}")
response2 = agent_executor.invoke({"input": user_query_2})
print(f"Agent: {response2['output']}")

# Query 3: A follow-up question leveraging conversational memory
# The agent should remember the previous query about 'Project Quantum Leap' from the internal docs.
user_query_3 = "Who is the project lead for the AI analytics platform mentioned earlier?"
print(f"\nUser: {user_query_3}")
response3 = agent_executor.invoke({"input": user_query_3})
print(f"Agent: {response3['output']}")

# Query 4: Another internal query, potentially requiring summarization or specific detail
user_query_4 = "Tell me more about the 'Synapse' product line. What makes it a significant revenue driver?"
print(f"\nUser: {user_query_4}")
response4 = agent_executor.invoke({"input": user_query_4})
print(f"Agent: {response4['output']}")

# Query 5: A query that might require both internal context and potentially external validation/expansion
user_query_5 = "What are InnovateCorp's core values, and how do they compare to industry best practices for tech companies in 2026?"
print(f"\nUser: {user_query_5}")
response5 = agent_executor.invoke({"input": user_query_5})
print(f"Agent: {response5['output']}")

print("\n--- InnovateCorp Knowledge Navigator Agent - Interaction End ---")
